In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Locate project root
project_root = Path.cwd().parents[2]

# Add project root to Python path
sys.path.insert(0, str(project_root))

# Import reusable data loader
from src.data_loader import load_cv_data

print("Project root:", project_root)
print("Setup completed.")

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Setup completed.


In [2]:
fold_datasets = load_cv_data(project_root)

for fold_number, (train_df, val_df) in enumerate(fold_datasets, start=1):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [3]:
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


# Convert skill IDs in every fold
for train_df, val_df in fold_datasets:
    train_df["skill_set"] = train_df["skill_id"].apply(convert_skill_id)
    val_df["skill_set"] = val_df["skill_id"].apply(convert_skill_id)

In [4]:
fold_train_df, fold_val_df = fold_datasets[0]

print("Skill ID:", fold_train_df["skill_id"].iloc[0])
print("Skill set:", fold_train_df["skill_set"].iloc[0])
print("Data type:", type(fold_train_df["skill_set"].iloc[0]))

print("\nMissing skill IDs:", fold_train_df["skill_id"].isna().sum())

Skill ID: 2_37_70
Skill set: {'37', '70', '2'}
Data type: <class 'set'>

Missing skill IDs: 11163


In [5]:
current_skills = {"A", "B", "C"}

previous_questions = [
    {"A", "B"},
    {"B", "C", "D"},
    {"C", "D"}
]

In [7]:
current_skills = {"A", "B", "C"}

previous_questions = [
    {"A", "B"},
    {"B", "C", "D"},
    {"C", "D"}
]

for previous_skills in previous_questions:
    overlap = current_skills & previous_skills

    binary_weight = 1 if overlap else 0

    fractional_weight = (
        len(overlap) / len(previous_skills)
        if previous_skills else 0
    )

    print(
        f"Previous: {previous_skills}, "
        f"Overlap: {overlap}, "
        f"Binary: {binary_weight}, "
        f"Fractional: {fractional_weight:.4f}"
    )

Previous: {'B', 'A'}, Overlap: {'B', 'A'}, Binary: 1, Fractional: 1.0000
Previous: {'D', 'B', 'C'}, Overlap: {'B', 'C'}, Binary: 1, Fractional: 0.6667
Previous: {'D', 'C'}, Overlap: {'C'}, Binary: 1, Fractional: 0.5000


In [8]:
outcomes = [1, 0, 1]

fractional_weights = [1, 2/3, 1/2]

weighted_correct = sum(
    y * w for y, w in zip(outcomes, fractional_weights)
)

weighted_experience = sum(fractional_weights)

weighted_accuracy = weighted_correct / weighted_experience

print("Weighted correct:", weighted_correct)
print("Weighted experience:", weighted_experience)
print("Weighted accuracy:", weighted_accuracy)

Weighted correct: 1.5
Weighted experience: 2.1666666666666665
Weighted accuracy: 0.6923076923076924


In [9]:

def add_same_skill_history_features(data, p0, alpha=10, training=False):
    """
    Generate binary and fractional same-skill history features.

    Binary:
        A previous question contributes 1 if at least one skill overlaps.

    Fractional:
        A previous question contributes:
        number of overlapping skills / number of skills in that question.

    Only previous interactions from the same student are considered.
    """

    df = data.copy()
    df = df.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    n = len(df)

    # Convert skill IDs into Python sets.
    skill_sets = df["skill_id"].apply(
        lambda x: set(str(x).split("_")) if pd.notna(x) else set()
    ).tolist()

    outcomes = df["correct"].to_numpy()

    # Calculate leakage-safe global prior.
    if training:
        total_correct = outcomes.sum()

        if n > 1:
            row_prior = (total_correct - outcomes) / (n - 1)
        else:
            row_prior = np.full(n, p0)
    else:
        row_prior = np.full(n, p0)

    # Arrays to store calculated features.
    binary_experience = np.zeros(n)
    binary_correct = np.zeros(n)

    fractional_experience = np.zeros(n)
    fractional_correct = np.zeros(n)

    # Process each student independently.
    for positions in df.groupby("user_id", sort=False).indices.values():

        # Previous question IDs associated with each skill.
        skill_question_ids = {}

        # Accumulated fractional experience for each skill.
        skill_fractional_experience = {}
        skill_fractional_correct = {}

        for pos in positions:

            current_skills = skill_sets[pos]

            # Calculate features BEFORE adding current question.
            matched_question_ids = set()

            for skill in current_skills:
                matched_question_ids.update(
                    skill_question_ids.get(skill, set())
                )

            # V2.1 / V2.3: Binary history.
            binary_experience[pos] = len(matched_question_ids)

            binary_correct[pos] = sum(
                outcomes[qid] for qid in matched_question_ids
            )

            # V2.2 / V2.4: Fractional history.
            fractional_experience[pos] = sum(
                skill_fractional_experience.get(skill, 0.0)
                for skill in current_skills
            )

            fractional_correct[pos] = sum(
                skill_fractional_correct.get(skill, 0.0)
                for skill in current_skills
            )

            # Add current question to history only AFTER calculation.
            if current_skills:
                weight = 1.0 / len(current_skills)

                for skill in current_skills:

                    skill_question_ids.setdefault(
                        skill, set()
                    ).add(pos)

                    skill_fractional_experience[skill] = (
                        skill_fractional_experience.get(skill, 0.0)
                        + weight
                    )

                    skill_fractional_correct[skill] = (
                        skill_fractional_correct.get(skill, 0.0)
                        + weight * outcomes[pos]
                    )

    # Store raw historical counts and weighted counts.
    df["same_skill_previous_interactions"] = binary_experience
    df["same_skill_previous_correct"] = binary_correct

    df["weighted_same_skill_previous_interactions"] = fractional_experience
    df["weighted_same_skill_previous_correct"] = fractional_correct

    # Smoothed accuracy features.
    df["same_skill_smoothed_accuracy"] = (
        binary_correct + alpha * row_prior
    ) / (binary_experience + alpha)

    df["weighted_same_skill_smoothed_accuracy"] = (
        fractional_correct + alpha * row_prior
    ) / (fractional_experience + alpha)

    # Log-transformed experience features.
    df["log_same_skill_previous_interactions"] = np.log1p(
        binary_experience
    )

    df["log_weighted_same_skill_previous_interactions"] = np.log1p(
        fractional_experience
    )

    return df


In [11]:
import importlib
import src.feature_engineering as fe

# Reload the latest saved version of the Python file
importlib.reload(fe)

# Check whether our function exists
print("Function available:", hasattr(fe, "add_same_skill_history_features"))

print("\nAvailable functions:")
print([name for name in dir(fe) if not name.startswith("_")])

Function available: True

Available functions:
['add_overall_history_features', 'add_same_skill_history_features', 'np', 'pd']


In [12]:
from src.feature_engineering import add_same_skill_history_features

print("Same-skill feature function imported successfully.")

Same-skill feature function imported successfully.


In [13]:
test_data = pd.DataFrame({
    "user_id": [1, 1, 1, 1],
    "order_id": [1, 2, 3, 4],
    "skill_id": ["A_B", "B_C_D", "C_D", "A_C"],
    "correct": [1, 0, 1, 0]
})

test_data

,user_id,order_id,skill_id,correct
0,1,1,A_B,1
1,1,2,B_C_D,0
2,1,3,C_D,1
3,1,4,A_C,0


In [14]:
test_features = add_same_skill_history_features(
    test_data,
    p0=0.5,
    alpha=10,
    training=False
)

test_features[[
    "order_id",
    "skill_id",
    "correct",
    "same_skill_previous_interactions",
    "same_skill_previous_correct",
    "weighted_same_skill_previous_interactions",
    "weighted_same_skill_previous_correct",
    "same_skill_smoothed_accuracy",
    "weighted_same_skill_smoothed_accuracy"
]]

,order_id,skill_id,correct,same_skill_previous_interactions,same_skill_previous_correct,weighted_same_skill_previous_interactions,weighted_same_skill_previous_correct,same_skill_smoothed_accuracy,weighted_same_skill_smoothed_accuracy
0,1,A_B,1,0.0,0.0,0.000000,0.0,0.500000,0.500000
1,2,B_C_D,0,1.0,1.0,0.500000,0.5,0.545455,0.523810
2,3,C_D,1,1.0,0.0,0.666667,0.0,0.454545,0.468750
3,4,A_C,0,3.0,2.0,1.333333,1.0,0.538462,0.529412


In [15]:
modified_test_data = test_data.copy()

# Change the current question's outcome
modified_test_data.loc[3, "correct"] = 1

modified_features = add_same_skill_history_features(
    modified_test_data,
    p0=0.5,
    alpha=10,
    training=False
)

columns_to_check = [
    "same_skill_previous_interactions",
    "same_skill_previous_correct",
    "weighted_same_skill_previous_interactions",
    "weighted_same_skill_previous_correct",
    "same_skill_smoothed_accuracy",
    "weighted_same_skill_smoothed_accuracy"
]

print("Original Q4 features:")
print(test_features.loc[3, columns_to_check])

print("\nModified Q4 features:")
print(modified_features.loc[3, columns_to_check])

print("\nAre Q4's features unchanged?")
print(
    np.allclose(
        test_features.loc[3, columns_to_check].astype(float),
        modified_features.loc[3, columns_to_check].astype(float)
    )
)

Original Q4 features:
same_skill_previous_interactions             3.000000
same_skill_previous_correct                  2.000000
weighted_same_skill_previous_interactions    1.333333
weighted_same_skill_previous_correct         1.000000
same_skill_smoothed_accuracy                 0.538462
weighted_same_skill_smoothed_accuracy        0.529412
Name: 3, dtype: float64

Modified Q4 features:
same_skill_previous_interactions             3.000000
same_skill_previous_correct                  2.000000
weighted_same_skill_previous_interactions    1.333333
weighted_same_skill_previous_correct         1.000000
same_skill_smoothed_accuracy                 0.538462
weighted_same_skill_smoothed_accuracy        0.529412
Name: 3, dtype: float64

Are Q4's features unchanged?
True
